# 04 — Per-spectrum investigation: visual signatures, the finder algorithm, and masking

So far you've worked with **catalogs as tables** — one row per absorber.
This notebook zooms back in to **one skewer at a time** and answers
three pedagogical questions that the table view glosses over:

1. **What does an LLS, a subDLA, and a DLA actually look like in
   `tau(v)` and `F = exp(-tau)`?**  We pick one clean example of each
   class and plot it.
2. **How are absorbers found?**  We walk through
   `hcd_analysis.catalog.find_systems_in_skewer` step-by-step on a real
   multi-absorber skewer, and see how the catalog gets built from raw
   `tau`.  Then we cover the two NHI-measurement modes (the fast COG
   estimator and the full Voigt fit).
3. **How are they masked?**  We compare three masking strategies on the
   same DLA: the catalog `pix_start..pix_end` (core only), the
   τ-space wing-aware mask (`hcd_analysis.masking`), and the PRIYA
   tau-based mask used in the production P1D pipeline.

This is the most code-heavy of the five notebooks because we're
literally retracing what `hcd_analysis` does to build the catalog.  Read
it next to `hcd_analysis/catalog.py` and `hcd_analysis/masking.py`.

**Where this fits in the analysis chain.**  NB01 told you what is in the
catalog at the row level.  NB04 tells you **how** that catalog row got
there: starting from a 1D τ array, what algorithm produces a
`(skewer_idx, pix_start, pix_end, NHI)` record?  And, once a record
exists, what does "masking the HCD" actually mean numerically?  These
two questions are the substrate of every science decision downstream.


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import h5py

DATA_ROOT    = Path(os.environ.get('HCD_LF_RAW_DIR', '<HCD_LF_RAW_DIR unset>'))
HCD_OUT_ROOT = Path(os.environ.get('HCD_OUTPUTS_DIR', '<HCD_OUTPUTS_DIR unset>'))
SIM  = 'ns0.81Ap1.6e-09herei3.59heref2.79alphaq1.71hub0.668omegamh20.145hireionz7.92bhfeedback0.0333'
SNAP = 22

SNAP_DIR = HCD_OUT_ROOT / SIM / f'snap_{SNAP:03d}'
RAW_FILE = DATA_ROOT / SIM / 'output' / f'SPECTRA_{SNAP:03d}' / 'lya_forest_spectra_grid_480.hdf5'

with open(SNAP_DIR / 'meta.json') as f:
    meta = json.load(f)
cat = np.load(SNAP_DIR / 'catalog.npz', allow_pickle=True)

dv      = float(meta['dv_kms'])
nbins   = int(meta['nbins'])
log_nhi = np.log10(cat['NHI'])
print(f'sim={SIM[:50]}…  snap={SNAP}  z={meta["z"]}  dv={dv:.3f} km/s  nbins={nbins}')
print(f'absorbers in catalog: {len(log_nhi):,}')


In [ ]:
# --- Tutorial figure-style preamble (auto-generated by _build_notebooks.py) ---
import matplotlib.pyplot as plt
from pathlib import Path as _Path

# Consistent style across all tutorial notebooks.
plt.rcParams.update({
    "font.size":        12,
    "axes.titlesize":   14,
    "axes.labelsize":   12,
    "xtick.labelsize":  11,
    "ytick.labelsize":  11,
    "legend.fontsize":  10,
    "figure.dpi":       110,
    "savefig.dpi":      200,
    "savefig.bbox":     "tight",
    "lines.linewidth":  1.4,
})

# Figures saved by this notebook land here.  Path is relative to the
# notebooks/tutorials/ directory; we resolve from __file__ for nbconvert
# and from cwd for an interactive Jupyter run.
try:
    _NB_DIR = _Path(__file__).parent
except NameError:
    _NB_DIR = _Path.cwd()
_FIG_DIR = _NB_DIR / "figures"
_FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name, *, tight=True):
    """Apply tight_layout, save to figures/<name>.png at dpi=200, return path.

    name should look like ``NB02_cddf_vs_ho21`` (no extension); the function
    appends ``.png`` and writes into ``notebooks/tutorials/figures/``.
    """
    if tight:
        fig.tight_layout()
    out = _FIG_DIR / f"{name}.png"
    fig.savefig(out, dpi=200)
    return out


## 1. Visual signatures of LLS, subDLA, DLA

We pick one **clean** example of each class — meaning the only absorber
on its sightline — so the visual signature isn't confused by neighbours.
The skewer indices below were found by enumerating the catalog;
recomputing them on the fly is left as an exercise.


In [ ]:
EXAMPLES = [
    # (class label, skewer_idx, pix_start, pix_end, log_NHI from catalog)
    ('LLS',    27,    35,   49, 17.42),
    ('subDLA', 184,  474,  510, 19.72),
    ('DLA',    210,  344,  409, 21.27),
]

# Stream just these three rows of tau (one h5py open, three slices).
with h5py.File(RAW_FILE, 'r') as f:
    tau_examples = {
        cls: f['tau/H/1/1215'][sk, :].astype(np.float64)
        for cls, sk, _, _, _ in EXAMPLES
    }

print('loaded tau for skewers:', list(tau_examples.keys()))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.0), sharex='col')
for col, (cls, sk, ps, pe, ln) in enumerate(EXAMPLES):
    tau_row = tau_examples[cls]
    F_row   = np.exp(-tau_row)
    v       = np.arange(len(tau_row)) * dv
    v_centre = 0.5 * (ps + pe) * dv
    # Window scales with absorber width so the DLA core does not fill the
    # whole panel.  At least ±1000 km/s (so the LLS panel still has
    # context); for the DLA panel this gives ~±2000 km/s.
    absorber_width_kms = (pe - ps + 1) * dv
    half = max(2.0 * absorber_width_kms, 1000.0)
    sel = (v > v_centre - half) & (v < v_centre + half)

    ax_t = axes[0, col]
    ax_t.plot(v[sel], tau_row[sel], color='C3', lw=0.9)
    ax_t.set_yscale('log')
    ax_t.axvspan(ps * dv, pe * dv, color='grey', alpha=0.2,
                 label='catalog pix range' if col == 0 else None)
    ax_t.axhline(100, color='k', linestyle=':', linewidth=0.8,
                 label=r'$\tau = 100$ (detect)' if col == 0 else None)
    ax_t.set_title(fr'{cls}: skewer {sk}, $\log N_{{\rm HI}}$ = {ln:.2f}')
    if col == 0:
        ax_t.set_ylabel(r'optical depth $\tau$')
        ax_t.legend(loc='upper right', fontsize=9)

    ax_f = axes[1, col]
    ax_f.plot(v[sel], F_row[sel], color='C0', lw=0.9)
    ax_f.axvspan(ps * dv, pe * dv, color='grey', alpha=0.2)
    ax_f.set_ylim(-0.05, 1.05)
    ax_f.set_xlabel('velocity along sightline  (km/s)')
    if col == 0:
        ax_f.set_ylabel(r'transmitted flux $F = e^{-\tau}$')

fig.suptitle(r'$\tau(v)$ (top) and $F(v)$ (bottom): one clean example per HCD class', y=1.00)
save_fig(fig, 'NB04_class_signatures')
plt.show()


**Visual signatures:**

* **LLS** (left).  Narrow, optically thick core (a few pixels of high
  τ), no resolvable damping wing.  In flux space the absorption is a
  brief dip back up to nearly the continuum.  The catalog window
  (`pix_start..pix_end`, grey band) almost exactly contains the absorber.

* **subDLA** (centre).  Wider absorption, deeper core, a hint of a
  damping wing visible on each side.  The catalog window still mostly
  contains it but the wings spill past the grey band.

* **DLA** (right).  Saturated black core spanning tens of pixels in F,
  surrounded by very prominent damping wings extending hundreds of km/s
  on each side.  **This is the key feature of a DLA**: most of the NHI
  information is in the wings, not the core.  The catalog
  `pix_start..pix_end` only captures the core (where τ > τ_threshold);
  any masking strategy that uses just that range *will* leave damping-
  wing power in the spectrum.  We come back to this in §4.


## 2. The finder algorithm, step by step

The catalog builder runs `find_systems_in_skewer` on every skewer.  It's
a 4-stage pipeline; here we walk through each stage on one real
multi-absorber skewer.

We use **skewer 1172** for this — it has three absorbers in the catalog:

| Catalog entry | pix_start | pix_end | log NHI | class |
|---|---|---|---|---|
| 1 | 374 | 410 | 20.19 | subDLA |
| 2 | 439 | 448 | 19.21 | subDLA |
| 3 | 598 | 613 | 17.45 | LLS |

Notice that entries 1 and 2 are only 28 pixels apart (≈ 280 km/s).
Watch how the merge step in stage 3 *almost* combines them, but doesn't
quite, because the gap exceeds the default `merge_dv_kms = 100 km/s`
(≈ 10 pixels at this dv).


In [ ]:
DEMO_SKEWER = 1172
with h5py.File(RAW_FILE, 'r') as f:
    tau_demo = f['tau/H/1/1215'][DEMO_SKEWER, :].astype(np.float64)
v_demo = np.arange(len(tau_demo)) * dv
print(f'skewer {DEMO_SKEWER}: max τ = {tau_demo.max():.2f}, '
      f'pixels above τ=100: {(tau_demo > 100).sum()}')


### Stage 1 — threshold

A pixel is a candidate for being inside an absorber if `τ > tau_threshold`.
The default `tau_threshold = 100` is far above any forest absorption
(typical forest τ < a few) and well below the peak τ of even an LLS
(typical peak τ ~ 10⁴–10⁹ for HCDs).  This is a coarse first pass; the
merging and Voigt-fitting stages refine it.


In [ ]:
TAU_THRESHOLD = 100.0
above = tau_demo > TAU_THRESHOLD

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(v_demo, tau_demo, color='C3', lw=0.8, label=r'$\tau(v)$ on demo skewer')
ax.fill_between(v_demo, 1e-3, tau_demo, where=above, color='red', alpha=0.3,
                label=fr'$\tau > {TAU_THRESHOLD:.0f}$ (above-threshold pixels)')
ax.axhline(TAU_THRESHOLD, color='k', linestyle=':', lw=0.8)
ax.set_yscale('log')
ax.set_ylim(1e-2, max(2.0, tau_demo.max() * 2))
ax.set_xlabel('velocity along sightline  (km/s)')
ax.set_ylabel(r'optical depth $\tau$')
ax.set_title(fr'Stage 1: above-threshold pixels (skewer {DEMO_SKEWER}, $\tau_{{\rm thr}} = {TAU_THRESHOLD:.0f}$)')
ax.legend(loc='upper right')
save_fig(fig, 'NB04_finder_stage1_threshold')
plt.show()


### Stage 2 — connected runs (with periodic-box trick)

The simulation box is periodic, so an absorber straddling pixel
`nbins-1` should connect with one starting at pixel `0`.  The trick used
in `find_systems_in_skewer` is to scan a *doubled* array
`[above | above]` for connected `True` runs — wrap-around runs then
appear as runs that cross the `nbins` boundary.

We reproduce the scan inline so you can see exactly what the catalog
builder does:


In [ ]:
def find_runs(boolean_array):
    '''Return list of (start, end_inclusive) for True runs.'''
    n = len(boolean_array)
    runs = []
    in_run = False
    start = 0
    for i, v in enumerate(boolean_array):
        if v and not in_run:
            start = i
            in_run = True
        elif not v and in_run:
            runs.append((start, i - 1))
            in_run = False
    if in_run:
        runs.append((start, n - 1))
    return runs

# Doubled array; keep only runs starting in [0, n)
above2 = np.concatenate([above, above])
all_runs = find_runs(above2)
runs = [(s, e) for (s, e) in all_runs if s < len(above)]

print(f'{len(runs)} connected runs found (before merging or length filter):')
for s, e in runs:
    wraps = e >= len(above)
    print(f'   pix {s:>5d} … {e:>5d}'
          f'  ({(e - s + 1):>3d} pixels)'
          f'{"   [WRAPS]" if wraps else ""}')


### Stage 3 — merge close runs

Two distinct above-threshold runs that are separated by fewer than
`merge_gap_pixels = round(merge_dv_kms / dv)` pixels are considered
parts of the same physical system (e.g. a DLA whose damping wings dip
just below the threshold for a couple of pixels in the middle).


In [ ]:
MERGE_DV_KMS = 100.0   # default in catalog.build_catalog
merge_gap_pixels = max(1, int(MERGE_DV_KMS / dv))
print(f'merge_gap_pixels = {merge_gap_pixels} (≈ {merge_gap_pixels * dv:.0f} km/s)')

merged = [runs[0]]
for s, e in runs[1:]:
    gap = s - merged[-1][1] - 1
    if gap <= merge_gap_pixels:
        print(f'   merging ({merged[-1][0]}, {merged[-1][1]}) + ({s}, {e})  [gap = {gap}]')
        merged[-1] = (merged[-1][0], e)
    else:
        merged.append((s, e))

print(f'{len(runs)} → {len(merged)} runs after merging.')
for s, e in merged:
    print(f'   pix {s:>5d} … {e:>5d}  ({(e - s + 1):>3d} pixels)')


### Stage 4 — drop short runs

Single- or double-pixel above-threshold blips are noise and discarded
(`min_pixels = 2` in production).  Whatever survives is what ends up in
the catalog.


In [ ]:
MIN_PIXELS = 2
final = [(s, e) for s, e in merged if (e - s + 1) >= MIN_PIXELS]

print(f'{len(merged)} → {len(final)} runs after length filter (min_pixels={MIN_PIXELS}).')

# Compare to what's actually in the catalog for this skewer.
on_demo = np.where(cat['skewer_idx'] == DEMO_SKEWER)[0]
print()
print(f'Catalog entries for skewer {DEMO_SKEWER}:')
for i in on_demo:
    print(f'   pix=({int(cat["pix_start"][i]):>4d}, {int(cat["pix_end"][i]):>4d})  '
          f'log NHI={np.log10(cat["NHI"][i]):.3f}  class='
          f'{"DLA" if log_nhi[i] >= 20.3 else "subDLA" if log_nhi[i] >= 19.0 else "LLS"}')
print()
print(f'Our reconstructed runs:')
for s, e in final:
    print(f'   pix=({s:>4d}, {e:>4d})')


The reconstructed pixel ranges agree with the catalog (to within 1 pixel
at the edges, where the threshold crossing is ambiguous).  What the
catalog adds beyond our reconstruction is the **NHI per system**, which
needs Voigt fitting and is the subject of the next section.


## 2b. NHI estimation: fast vs Voigt

Once a system's pixel range is known, we have to turn its `tau` profile
into a column density.  Two estimators are available:

* **Fast estimator** (`hcd_analysis.voigt_utils.nhi_from_tau_fast`):
  applies the Ladenburg–Reiche sum rule directly to the τ array:
  `N_HI = Σ τ_i · dv / σ_int`.  This is **exact** for a noise-free,
  complete absorption profile — the integrated absorption is a fixed
  atomic-physics constant regardless of line shape.  The only error
  source is **truncation** of pixels below `τ_threshold`, and that
  truncation costs **< 0.005 dex** even in the worst case (an LLS at
  the detection boundary with extreme thermal broadening); for any
  log NHI ≥ 18 it's < 0.002 dex, and for DLAs it's immeasurable.
  Full derivation: `docs/fast_mode_physics.md`.  This is what the
  production LF run uses.
* **Voigt fit** (`fit_nhi_from_tau`): expands the window by ±200 pixels
  and fits a single Voigt profile in log-space.  Recovers the same NHI
  as fast mode to < 0.001 dex on clean profiles (both are measuring
  `∫τdv`, just by different routes), but **also returns `b`** and is
  more robust when a single nearby low-τ absorber contaminates the
  fast-mode integration window.  Used by the HiRes run, which has the
  SNR to make `b` measurable.

The two estimators **agree** on clean DLAs.  The Voigt fit's real
advantage isn't accuracy at high NHI — it's giving you `b` and being
forgiving about a single contaminating absorber in the window.  We
demonstrate both on the DLA from §1 below.

**Voigt-profile constants** (you'll need these for the damping-wing
concept-check question in the tutorial README):

| Symbol | Value | Meaning |
|---|---|---|
| `λ_α` | 1215.67 Å | Lyα rest wavelength |
| `ν_α` | `c / λ_α` ≈ 2.466 × 10¹⁵ Hz | Lyα rest frequency |
| `f_α` | 0.4164 | Lyα oscillator strength |
| `γ_α` | 6.265 × 10⁸ s⁻¹ | Lyα natural damping rate (full Einstein-A coefficient A₂₁; the Lorentzian wing uses γ_α directly, *not* γ_α/2π) |
| `σ_α,int` | `πe²f_α / (m_e c)` ≈ 0.0110 cm² · Hz | line-integrated cross section |

**Lorentzian damping-wing formula** (valid for `|Δv| ≫ b`, i.e. far
enough from line center that the Doppler core is no longer relevant):

$$
\tau_{\rm wing}(\Delta v)
\;\approx\;
\frac{N_{\rm HI}\, \sigma_{\alpha,{\rm int}}\, \gamma_{\alpha}}
     {4\pi^{2}\,(\nu_{\alpha}\,\Delta v/c)^{2}}
\;=\;
\frac{N_{\rm HI}\, \sigma_{\alpha,{\rm int}}\, \gamma_{\alpha}\, c^{2}}
     {4\pi^{2}\,\nu_{\alpha}^{2}\,\Delta v^{2}}.
$$

Solving `τ_wing = 1` for `Δv` gives the wing-saturation radius — the
velocity offset where the absorption stops being optically thick.
For a `log N_HI = 21`, `b = 25 km/s` DLA this works out to about
**±1600 km/s** (well outside the catalog's `(pix_start, pix_end)` core,
which is why §2b widens the fit window).  Plug your numbers into the
cell below to verify against one of the real DLAs in this snap.


In [ ]:
# Voigt / damping-wing constants
LAMBDA_ALPHA  = 1215.67e-8       # cm
NU_ALPHA      = 2.998e10 / LAMBDA_ALPHA          # Hz
F_ALPHA       = 0.4164
GAMMA_ALPHA   = 6.265e8          # s^-1
SIGMA_INT     = 0.01103          # cm^2 * Hz   (pi e^2 f / (m_e c))


def tau_wing_lorentz(nhi, dv_kms):
    '''Lorentzian-wing tau (valid for |dv_kms| >> b).  dv_kms can be array-like.'''
    dnu_Hz = NU_ALPHA * (np.asarray(dv_kms) * 1e5) / 2.998e10  # Δv in cm/s → Δν in Hz
    return nhi * SIGMA_INT * GAMMA_ALPHA / (4.0 * np.pi**2 * dnu_Hz**2)


# Concept-check helper: where does tau drop to 1 for a chosen NHI?
def dv_crit_km_s(nhi_test):
    '''Solve tau_wing = 1 for |dv| in km/s.'''
    return np.sqrt(nhi_test * SIGMA_INT * GAMMA_ALPHA / (4.0 * np.pi**2)) \
           * 2.998e10 / NU_ALPHA / 1e5

for log_nhi in (20.3, 21.0, 21.5):
    print(f'log NHI = {log_nhi}:  |Δv| where τ=1  ≈  {dv_crit_km_s(10**log_nhi):7.1f}  km/s')


In [ ]:
from hcd_analysis.voigt_utils import nhi_from_tau_fast, fit_nhi_from_tau, tau_voigt

# DLA from §1: skewer 210, pix=(344, 409), catalog log NHI = 21.27
DLA_SK, DLA_PS, DLA_PE = 210, 344, 409

# Pull the row.
with h5py.File(RAW_FILE, 'r') as f:
    tau_dla = f['tau/H/1/1215'][DLA_SK, :].astype(np.float64)

# (a) Fast estimator on the core only
core = tau_dla[DLA_PS:DLA_PE + 1]
NHI_fast = nhi_from_tau_fast(core, dv_kms=dv)

# (b) Voigt fit on the wider window (±200 pixels around the core)
WING_PIX = 200
lo = max(0, DLA_PS - WING_PIX)
hi = min(len(tau_dla) - 1, DLA_PE + WING_PIX)
seg     = tau_dla[lo:hi + 1]
peak_i  = int(np.argmax(seg))
v_seg   = (np.arange(len(seg)) - peak_i) * dv
NHI_fit, b_fit, ok = fit_nhi_from_tau(seg, v_seg)

print(f'Catalog log NHI    = 21.27   (the production fast-mode value)')
print(f'Fast estimator     = {np.log10(NHI_fast):.3f}   (recomputed here on core only)')
print(f'Voigt fit          = {np.log10(NHI_fit):.3f}    (b = {b_fit:.1f} km/s, success={ok})')


In [ ]:
# Plot the data and the Voigt-fit model on the same window
v_full = np.arange(len(tau_dla)) * dv
v_lo, v_hi = lo * dv, hi * dv
v_peak = (lo + peak_i) * dv

# Voigt model evaluated on the full velocity grid relative to peak
tau_model = tau_voigt(v_full - v_peak, NHI=NHI_fit, b_kms=b_fit)

fig, ax = plt.subplots(figsize=(11, 4.3))
ax.plot(v_full, tau_dla, color='C3', lw=0.9, label=r'data: $\tau(v)$ from sim')
ax.plot(v_full, tau_model, color='k', lw=1.2, linestyle='--',
        label=fr'Voigt fit  $\log N_{{\rm HI}}$ = {np.log10(NHI_fit):.2f}, $b$ = {b_fit:.0f} km/s')
ax.axvspan(DLA_PS * dv, DLA_PE * dv, color='grey', alpha=0.2, label='catalog pix range (core only)')
ax.axvspan(v_lo, v_hi, color='C0', alpha=0.07, label=fr'Voigt fit window ($\pm$ {WING_PIX} pix)')
ax.set_yscale('log')
ax.set_ylim(1e-2, max(2.0, tau_dla.max() * 2))
ax.set_xlim(v_lo - 200, v_hi + 200)
ax.set_xlabel('velocity along sightline  (km/s)')
ax.set_ylabel(r'optical depth $\tau$')
ax.set_title(fr'DLA on skewer {DLA_SK}: $\tau$ data and Voigt model (note damping-wing match)')
ax.legend(loc='upper right')
save_fig(fig, 'NB04_voigt_fit_one_dla')
plt.show()


**What you should see.**  The Voigt model tracks both the saturated
core (where any model would fit by saturation alone) AND the damping
wings on each side.  The wings extend several hundred km/s past the
catalog's grey band — this is why fitting *only* the core
underestimates NHI for DLAs, and why a serious P1D analysis must mask
the wings as well as the core.


## 3. Three masking strategies, side by side

Now the practical question: how do you remove an HCD's contribution to
the flux power spectrum?  The package provides three strategies, each
embodying a different definition of "where the absorber is":

| Strategy | What's masked | Implementation |
|---|---|---|
| `pixrange` | Catalog `pix_start..pix_end` (the τ > 100 core only) | `hcd_analysis.masking.build_skewer_mask` |
| `tauspace` | Walk outward from each system's τ-peak until τ drops below `wing_threshold[class] + τ_eff`.  Wing thresholds: 0.25 (DLA), 0.50 (subDLA), 1.00 (LLS). | `build_skewer_mask_from_tau` |
| `priya`    | Same wing-walk algorithm but starts from the global τ-peak of the sightline; only triggers if `max(τ) > 10⁶`.  This is the **production** P1D mask. | `priya_dla_mask_row` |

We apply all three to the DLA from §2 and overlay the masks on `F(v)`.


In [ ]:
from hcd_analysis.catalog import AbsorberCatalog
from hcd_analysis.masking import (
    build_skewer_mask,
    build_skewer_mask_from_tau,
    priya_dla_mask_row,
    DEFAULT_WING_THRESHOLD,
)

# We need an AbsorberCatalog for the package mask builders.  Slice it down
# to just the absorbers on our DLA skewer.
ac = AbsorberCatalog.load_npz(SNAP_DIR / 'catalog.npz')
abs_on_skewer = [a for a in ac.absorbers if a.skewer_idx == DLA_SK]
print(f'absorbers on skewer {DLA_SK}: {len(abs_on_skewer)}')
for a in abs_on_skewer:
    print(f'   class={a.absorber_class:6s}  pix=({a.pix_start:>4d}, {a.pix_end:>4d})  '
          f'log NHI={a.log_NHI:.2f}')

# Compute τ_eff = -ln(<F>) on this skewer (for a real run we'd use the
# global <F> from the unmasked file; here we use this skewer's <F> as a
# stand-in to keep the demo self-contained).
F_dla = np.exp(-tau_dla)
tau_eff = -np.log(max(F_dla.mean(), 1e-30))
print(f'\ntau_eff (this-skewer proxy): {tau_eff:.4f}')


In [ ]:
# Three masks
mask_pixrange = build_skewer_mask(len(tau_dla), abs_on_skewer)
mask_tauspace = build_skewer_mask_from_tau(
    tau_dla, abs_on_skewer, tau_eff,
    wing_threshold_by_class=DEFAULT_WING_THRESHOLD,
)
mask_priya = priya_dla_mask_row(tau_dla, tau_eff,
                                tau_dla_detect=1e6, tau_mask_scale=0.25)
if mask_priya is None:
    mask_priya = np.zeros_like(tau_dla, dtype=bool)

print(f'pixels masked:  pixrange={mask_pixrange.sum():>4d}   '
      f'tauspace={mask_tauspace.sum():>4d}   priya={mask_priya.sum():>4d}')


In [ ]:
v_full = np.arange(len(tau_dla)) * dv

fig, axes = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True)
labels = [
    ('pixrange  (catalog core only)',         mask_pixrange, 'C0'),
    ('tauspace  (wing-aware, per-class thr.)', mask_tauspace, 'C1'),
    ('priya     (production DLA mask)',        mask_priya,    'C2'),
]
for ax, (lbl, mk, c) in zip(axes, labels):
    ax.plot(v_full, F_dla, color='black', lw=0.7, label=r'$F(v)$')
    # Show masked region as shaded.  fill_between with where=mask.
    ax.fill_between(v_full, 0, 1.05, where=mk, color=c, alpha=0.35,
                    label=f'{lbl}: {mk.sum()} pix masked')
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel(r'transmitted flux $F$')
    ax.legend(loc='lower right')

axes[0].set_title(f'DLA on skewer {DLA_SK}: three masking strategies on the same flux')
axes[-1].set_xlabel('velocity along sightline  (km/s)')
# Zoom around the absorber for clarity
v_centre = 0.5 * (DLA_PS + DLA_PE) * dv
for ax in axes:
    ax.set_xlim(v_centre - 2500, v_centre + 2500)
save_fig(fig, 'NB04_three_masks')
plt.show()


**Reading the figure.**  All three masks cover the same saturated core,
but they differ on what to do with the wings:

* `pixrange` (top) is the narrowest — it only covers the τ > 100 core
  and leaves the entire damping-wing absorption visible.  Using this
  mask in a P1D estimator leaks damping-wing power into the small-k
  tail.  The package keeps `pixrange` only for regression testing and
  diagnostic purposes — **do not use it for science**.
* `tauspace` (middle) walks outward from each system's peak until τ
  drops below `0.25 + τ_eff` (for a DLA), capturing the full damping
  wings.  This is the strategy used for class-selective masking in the
  diagnostic P1D variants.
* `priya` (bottom) does almost the same wing-walk but from the
  *global* τ-peak of the sightline, and only kicks in if the sightline
  has any pixel with τ > 10⁶ (i.e. contains a DLA).  This is the
  production mask used in `compute_p1d_priya_masked` and is what feeds
  the emulator's `no_DLA_priya` variant.  The `τ > 10⁶` trigger
  threshold corresponds to log NHI ≈ 20.3 for typical Doppler widths
  and is the PRIYA recipe described in Bird et al. 2023 §2.7
  (arXiv:2306.05471).


## 3b. Three fill strategies

Once you've decided which pixels to mask, you still have to decide
*what* to put in them.  The package supports three fills, all in
`hcd_analysis.masking`:

| Strategy | What goes in masked pixels | Effect on δF |
|---|---|---|
| `zero_tau`   | τ = 0  → F = 1            | huge positive δF spike (unphysical) |
| `mean_flux`  | τ = τ_eff  → F = ⟨F⟩      | δF = 0 in masked region (PRIYA recipe) |
| `contiguous` | log-τ linear interpolation| smooth bridge across the mask |

The PRIYA P1D pipeline uses `mean_flux` because it neutralises the
masked region without injecting large-scale power.  `zero_tau` is
useful only as a sanity check.  `contiguous` is occasionally useful for
narrow LLS masks where you want continuity but it can underestimate
mean flux when the mask is wide.

**Production default for this project: `mask = priya` + `fill = mean_flux`.**
That's the pair used by `compute_p1d_priya_masked` and the
`no_DLA_priya` variant the upstream emulator consumes.  Any new
ablation should be reported relative to this baseline.


In [ ]:
from hcd_analysis.masking import _FILL_FUNCTIONS

# Use the tauspace mask we built above and apply each fill strategy
tau_filled = {
    name: fn(tau_dla, mask_tauspace)
    for name, fn in _FILL_FUNCTIONS.items()
}
F_filled = {name: np.exp(-t) for name, t in tau_filled.items()}

fig, ax = plt.subplots(figsize=(11, 4.3))
ax.plot(v_full, F_dla, color='black', lw=0.7, label='unmasked')
for name, F in F_filled.items():
    ax.plot(v_full, F, lw=1.2, label=f'fill = {name}')
ax.axvspan(v_full[mask_tauspace].min(),
           v_full[mask_tauspace].max(),
           color='C1', alpha=0.15, label='mask region (tauspace)')
ax.set_xlim(v_centre - 2500, v_centre + 2500)
ax.set_ylim(-0.05, 1.10)
ax.set_xlabel('velocity along sightline  (km/s)')
ax.set_ylabel(r'transmitted flux $F$')
ax.set_title('Three fill strategies applied to the same tauspace mask')
ax.legend(loc='lower right')
save_fig(fig, 'NB04_three_fills')
plt.show()


**What you should see.**

* `zero_tau` (yellow-ish) bumps F up to 1.0 inside the mask — clearly
  wrong, and exactly why this strategy is for diagnostics only.
* `mean_flux` sits at F = ⟨F⟩ ≈ 0.85 across the mask — gentle, no
  spikes.  This is what production P1D pipelines use.
* `contiguous` linearly bridges the boundary values — visually smooth,
  but it can sneak in extra δF if the wing endpoints are very different.


## Suggested student exercises

1. **Reproduce the three-class figure for a different sim, snap, or
   redshift.**  Pick z = 3 (snap 017) and find one clean LLS / subDLA /
   DLA each.  Does the relative damping-wing prominence change with z?
2. **Re-run `find_systems_in_skewer` with different parameters.**  Try
   `tau_threshold = 50` (more sensitive, more spurious LLSs) and
   `tau_threshold = 1000` (less sensitive, miss weak LLSs).  Plot the
   per-class catalog count as a function of `tau_threshold`.
3. **Compare the fast estimator and Voigt fit on a sample of 100
   DLAs.**  Compute the residual `log(NHI_voigt) − log(NHI_fast)` and
   plot its distribution.  This quantifies the systematic bias of
   fast-mode HCD catalogs.
4. **Compute the masked P1D for one sim under each of the three masks
   above.**  Use `compute_p1d_single` with `mask_scheme="pixrange"`,
   then `mask_scheme="tauspace"`, then `compute_p1d_priya_masked`.
   Plot the three P1Ds together — the spread between them is the
   "masking systematic".

## Where to next

* If you want to keep digging into HCD identification: read
  `hcd_analysis/catalog.py` (the catalog builder we just walked
  through) and `hcd_analysis/voigt_utils.py` (the Voigt fitter).
* If you want to dig into mask-induced P1D systematics: read
  `docs/masking_strategy.md` and `scripts/run_test10.py`.
* If you're ready to move past per-spectrum work and start thinking
  about emulator inputs: notebooks 02 and 03 are the population-level
  view, and the next milestone is the phase-1 emulator-data cache.
